# BBF202E Numerical Methods in Computer Engineering
## Spring 2025–2026 Term Project
### Robust Spatio-Temporal Data Imputation and Nowcasting in Agricultural Sensor Networks via Iterative SVD

### Name : Atillahan Gözütok
### Student ID : 150240332
---

## Imports & Global Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Exactly four features required by the project specification (Task 0)
FEATURES = [
    "sicaklik_2m_mean",
    "nispi_nem_2m_mean",
    "gunes_radyasyon",
    "ruzgar_hizi",
]

# Human-readable labels (used in plots)
FEATURE_LABELS = [
    "Temperature\n(sicaklik_2m_mean)",
    "Humidity\n(nispi_nem_2m_mean)",
    "Solar Radiation\n(gunes_radyasyon)",
    "Wind Speed\n(ruzgar_hizi)",
]

print("Libraries loaded successfully.")

---
## Task 0 – Data Interface and Feature Selection

The function below loads an arbitrary number of station CSV files, keeps only the four required features, aligns them on the `zaman_olcum` timestamp column, and returns a merged DataFrame.

Key design decisions:
- No station names or row counts are hard-coded -> compatible with the Blind Test.
- Duplicate timestamps within a single station are dropped before merging.
- An outer join is used so that rows present in at least one station are retained.
- Rows where all values across all stations are NaN are discarded.

In [ ]:
def load_and_align(paths: list[str]) -> pd.DataFrame:
    """
    Load CSV files for an arbitrary number of stations, keep only the four
    required features, align them on 'zaman_olcum', and return the merged
    DataFrame with 4 * n_stations columns.

    The function is intentionally agnostic to station IDs for generalization.
    """
    dfs = []
    for idx, path in enumerate(paths):
        df = pd.read_csv(path, parse_dates=["zaman_olcum"])

        # Drop exact duplicate timestamps within one station
        df = df.drop_duplicates(subset="zaman_olcum")
        df = df.set_index("zaman_olcum")[FEATURES]

        # Suffix each column with the station index so columns stay unique
        df.columns = [f"{col}_s{idx+1}" for col in FEATURES]
        dfs.append(df)

    # Outer-join on timestamp
    merged = dfs[0].join(dfs[1:], how="outer")

    # Drop rows where EVERY value across ALL stations is NaN
    merged = merged.dropna(how="all")
    merged.sort_index(inplace=True)
    return merged

---
## Task 1 – Preprocessing and Standardisation

Standardisation to zero mean and unit variance is mandatory before SVD:
- Variables with large magnitudes (e.g. Solar Radiation) would otherwise dominate the singular values, making reconstruction of smaller-scale variables impossible.
- The mean and standard deviation are computed only from observed (not NaN) entries to avoid bias.

In [ ]:
def standardise(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Standardise each column to zero mean and unit variance.
    Statistics are computed only from observed (not NaN) entries.

    Returns
    -------
    X_std    : standardised matrix (NaNs preserved)
    col_mean : per-column means of observed values
    col_std  : per-column standard deviations of observed values
    """
    col_mean = np.nanmean(X, axis=0)
    col_std  = np.nanstd(X,  axis=0)
    col_std[col_std == 0] = 1.0          # for zero variance columns
    X_std = (X - col_mean) / col_std
    return X_std, col_mean, col_std


def unstandardise(X_std: np.ndarray,
                  col_mean: np.ndarray,
                  col_std: np.ndarray) -> np.ndarray:
    """Reverse standardisation to recover original-scale values."""
    return X_std * col_std + col_mean

---
## Task 2 – SVD from Scratch

#### The decomposition is implemented with the Power Method with Deflation

### Algorithm outline
1. Form $C = M^\top M$
2. Extract eigenpairs of $C$ one at a time using the Power Method; deflate after each extraction.
   - Eigenvalues of $C$ = squared singular values of $M$.
3. Recover left singular vectors: $u_i = M v_i / \sigma_i$.
4. Fix sign ambiguity: enforce that the largest-magnitude entry of each $u_i$ is positive.


The Power Method returns eigenvectors defined up to a global sign flip. We resolve this with the largest-entry convention: the sign of $u_i$ (and correspondingly $v_i$) is chosen so that `u_i[argmax(|u_i|)] > 0`.

In [ ]:
def _power_iteration(A: np.ndarray,
                     num_iter: int = 200,
                     tol: float = 1e-10) -> tuple[np.ndarray, float]:
    """
    Find the dominant eigenvector and eigenvalue of a symmetric PSD matrix A
    using the Power Method with convergence detection.

    Convergence is declared when the Rayleigh quotient changes by less than
    `tol` between successive iterations.
    """
    n = A.shape[0]
    rng = np.random.default_rng(42)       # fixed seed for reproducible results
    v = rng.standard_normal(n)
    v /= np.linalg.norm(v)

    eigenvalue = 0.0
    for _ in range(num_iter):
        Av = A @ v
        eigenvalue_new = v @ Av           # Rayleigh quotient estimate
        v_new = Av / (np.linalg.norm(Av) + 1e-300)
        if abs(eigenvalue_new - eigenvalue) < tol:
            eigenvalue = eigenvalue_new
            v = v_new
            break
        eigenvalue, v = eigenvalue_new, v_new

    return v, float(eigenvalue)


def my_svd(M: np.ndarray,
           full_matrices: bool = False) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Economy SVD  M = U Σ V^T  via eigendecomposition with
    Power-Method + Deflation.

    Parameters
    ----------
    M : (n, p) real matrix

    Returns
    -------
    U     : (n, k) left singular vectors
    sigma : (k,)   singular values in descending order
    Vt    : (k, p) right singular vectors (transposed, numpy convention)
    """
    n, p = M.shape
    k = min(n, p)          # maximum number of singular triplets

    C = M.T @ M            # p × p Gram matrix

    V       = np.zeros((p, k))
    sigma   = np.zeros(k)
    U       = np.zeros((n, k))
    C_deflated = C.copy()

    for i in range(k):
        v_i, lam_i = _power_iteration(C_deflated)

        # Clamp tiny negatives caused by floating-point errors
        lam_i = max(lam_i, 0.0)
        sig_i = np.sqrt(lam_i)

        if sig_i < 1e-12:
            # Remaining singular values are numerically zero
            k = i
            break

        # Left singular vector:  u_i = M v_i / σ_i
        u_i = M @ v_i / sig_i

        # Fix sign ambiguity using the largest-entry convention
        if u_i[np.argmax(np.abs(u_i))] < 0:
            u_i = -u_i
            v_i = -v_i

        V[:, i]  = v_i
        sigma[i] = sig_i
        U[:, i]  = u_i

        # Deflation: remove the contribution of this eigenpair
        C_deflated -= lam_i * np.outer(v_i, v_i)

    # Trim to the actual rank found
    V     = V[:, :k]
    sigma = sigma[:k]
    U     = U[:, :k]

    return U, sigma, V.T          # V.T matches numpy convention

### Orthogonality Verification

A correct SVD satisfies $U^\top U \approx I$ and $V^\top V \approx I$. The cell below checks the maximum off-diagonal entry of both products and reports PASS / FAIL against a tolerance of $10^{-4}$.

In [ ]:
def verify_orthogonality(U: np.ndarray, Vt: np.ndarray, tol: float = 1e-4) -> None:
    """
    Check U^T U ≈ I  and  V^T V ≈ I.
    Prints the max off-diagonal absolute error for both matrices.
    """
    UtU = U.T @ U
    VtV = Vt @ Vt.T      # Vt is V^T, so Vt @ Vt.T = V^T V

    k = UtU.shape[0]
    mask = ~np.eye(k, dtype=bool)

    err_U = np.max(np.abs(UtU[mask])) if k > 1 else 0.0
    err_V = np.max(np.abs(VtV[mask])) if k > 1 else 0.0

    print(f"  Orthogonality | U^T U: max off-diag = {err_U:.2e}"
          f"  (tol={tol})  {'PASS' if err_U < tol else 'FAIL'}")
    print(f"  Orthogonality | V^T V: max off-diag = {err_V:.2e}"
          f"  (tol={tol})  {'PASS' if err_V < tol else 'FAIL'}")

---
## Task 3 – Iterative Singular Value Thresholding (SVT)

The SVT algorithm iteratively completes a matrix with missing entries:

| Step | Description |
|------|-------------|
| **A** | Fill NaN positions with **0** . |
| **B** | Decompose with `my_svd`; keep only the **top-r** singular triplets. |
| **C** | Update **only missing entries** with the low-rank reconstruction; observed entries are held fixed. |
| **D** | Repeat until $\|X_{k+1} - X_k\|_F < \varepsilon$ (threshold $\varepsilon = 10^{-4}$). |

In [ ]:
def svt_impute(X_missing: np.ndarray,
               rank: int,
               max_iter: int = 500,
               eps: float = 1e-4) -> np.ndarray:
    """
    Iterative Singular Value Thresholding for matrix completion.

    Parameters
    ----------
    X_missing : N x p matrix with NaN at missing positions (standardised)
    rank      : number of singular values to retain (low-rank truncation)
    max_iter  : maximum SVT iterations
    eps       : Frobenius-norm convergence threshold

    Returns
    -------
    X_filled : completed matrix on the standardised scale
    """
    nan_mask = np.isnan(X_missing)          # True where data is missing
    X_filled = X_missing.copy()
    X_filled[nan_mask] = 0.0               # Step A: zero-fill NaNs

    X_prev = X_filled.copy()

    for iteration in range(1, max_iter + 1):
        # Step B: low-rank reconstruction via custom SVD
        U, sigma, Vt = my_svd(X_prev)

        r = min(rank, len(sigma))
        U_r     = U[:, :r]
        sigma_r = sigma[:r]
        Vt_r    = Vt[:r, :]

        X_recon = U_r * sigma_r @ Vt_r    # = U_r diag(sigma_r) Vt_r

        # Step C: update missing entries only; keep observed entries intact
        X_filled[nan_mask] = X_recon[nan_mask]

        # Step D: Frobenius-norm convergence check
        diff = np.linalg.norm(X_filled - X_prev, 'fro')
        if diff < eps:
            print(f"    SVT converged at iteration {iteration}  (Δ={diff:.2e})")
            break

        X_prev = X_filled.copy()

    else:
        print(f"    SVT reached max_iter={max_iter}  (Δ={diff:.2e})")

    return X_filled

---
## Helper Utilities – Masking, RMSE, Scenario Generators

In [ ]:
def rmse(true: np.ndarray, pred: np.ndarray) -> float:
    """Root Mean Squared Error over all elements (NaN-safe)."""
    return float(np.sqrt(np.nanmean((true - pred) ** 2)))


def rmse_feature(true: np.ndarray,
                 pred: np.ndarray,
                 col_indices: list[int],
                 label: str) -> float:
    """RMSE for a subset of columns (identified by index)."""
    t = true[:, col_indices].ravel()
    p = pred[:, col_indices].ravel()
    r = float(np.sqrt(np.nanmean((t - p) ** 2)))
    print(f"    RMSE [{label:30s}] = {r:.4f}")
    return r


def inject_random_missing(X_clean: np.ndarray,
                          station_cols: list[int],
                          fraction: float = 0.20,
                          seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """
    Scenario A - Random: randomly blank `fraction` of entries in the
    columns belonging to one station.
    Returns the masked matrix and a boolean mask (True = artificially removed).
    """
    rng = np.random.default_rng(seed)
    X_masked = X_clean.copy()
    mask = np.zeros_like(X_clean, dtype=bool)
    for c in station_cols:
        col = X_clean[:, c]
        valid_rows = np.where(~np.isnan(col))[0]
        n_remove   = max(1, int(len(valid_rows) * fraction))
        chosen     = rng.choice(valid_rows, size=n_remove, replace=False)
        X_masked[chosen, c] = np.nan
        mask[chosen, c]     = True
    return X_masked, mask


def inject_burst_missing(X_clean: np.ndarray,
                         station_cols: list[int],
                         hours: int,
                         seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """
    Scenarios B & C - Burst / Blackout: blank a contiguous block of `hours`
    rows in the station's columns, starting from a random valid position.
    """
    rng = np.random.default_rng(seed)
    N = X_clean.shape[0]
    X_masked = X_clean.copy()
    mask = np.zeros_like(X_clean, dtype=bool)
    start = int(rng.integers(0, max(1, N - hours)))
    end   = min(start + hours, N)
    for c in station_cols:
        X_masked[start:end, c] = np.nan
        mask[start:end, c]     = True
    return X_masked, mask

---
## Task 4 – Stress Test Experiments

Three missingness scenarios are evaluated:

| Scenario | Description |
|----------|-------------|
| **A – Random 20%** | 20% of entries removed at random for Station 1. |
| **B – Burst 24h** | Contiguous 24-hour block removed for Station 1. |
| **C – Blackout 72h** | Contiguous 72-hour block removed for Station 1. |

For each scenario, overall RMSE, temperature RMSE, and wind speed RMSE are reported separately.

In [ ]:
def run_stress_tests(X_std_clean: np.ndarray,
                     col_mean: np.ndarray,
                     col_std: np.ndarray,
                     n_stations: int,
                     rank: int) -> dict:
    """
    Mandatory stress-test experiments.
    Returns a dict of RMSE values for every scenario x feature combination.
    """
    results = {}
    n_feat = len(FEATURES)

    # Station 0 columns used as the probe station
    station0_cols = list(range(n_feat))

    scenarios = {
        "A_random_20pct":  ("random", {}),
        "B_burst_24h":     ("burst",  {"hours": 24}),
        "C_blackout_72h":  ("burst",  {"hours": 72}),
    }

    print("=" * 60)
    print("TASK 4 - STRESS TESTS")
    print("=" * 60)

    for name, (kind, kw) in scenarios.items():
        print(f"\n  Scenario {name}")
        if kind == "random":
            X_masked, mask = inject_random_missing(X_std_clean, station0_cols)
        else:
            X_masked, mask = inject_burst_missing(X_std_clean, station0_cols, **kw)

        X_imputed = svt_impute(X_masked, rank=rank)

        # Evaluate only at artificially removed positions
        true_vals    = X_std_clean[mask]
        imputed_vals = X_imputed[mask]
        overall_rmse = float(np.sqrt(np.nanmean((true_vals - imputed_vals) ** 2)))
        print(f"    Overall RMSE = {overall_rmse:.4f}  (standardised scale)")

        def feature_rmse(feat_col_name, feat_idx_in_station):
            col = station0_cols[feat_idx_in_station]
            row_mask = mask[:, col]
            t = X_std_clean[row_mask, col]
            p = X_imputed[row_mask, col]
            r = float(np.sqrt(np.nanmean((t - p) ** 2)))
            print(f"    RMSE [{feat_col_name:30s}] = {r:.4f}")
            return r

        r_temp = feature_rmse("sicaklik_2m_mean (Temperature)", 0)
        r_wind = feature_rmse("ruzgar_hizi      (Wind Speed)",  3)

        results[name] = {
            "overall": overall_rmse,
            "temperature_rmse": r_temp,
            "wind_rmse": r_wind,
        }

    return results

---
## Variable Analysis (Report Section A)

Each of the four sensor types is isolated and imputed independently across three gap regimes (3h / 24h / 72h). This reveals which physical variables SVD reconstructs most confidently.

In [ ]:
def variable_analysis(X_std_clean: np.ndarray,
                      rank: int,
                      gap_lengths_hours: list[int] | None = None,
                      n_trials: int = 5) -> dict:
    """
    Compare SVT imputation quality across all four sensor types,
    under three gap-length regimes (short / medium / long burst).

    For each gap length and each feature:
      1. Artificially remove that feature's column for station 0.
      2. Impute with SVT (all 12 columns except the target).
      3. Record RMSE at the removed positions.

    Returns
    -------
    results : {gap_h: {feat_label: {"mean": float, "std": float}}}
    """
    if gap_lengths_hours is None:
        gap_lengths_hours = [3, 24, 72]

    results = {g: {} for g in gap_lengths_hours}

    print("=" * 60)
    print("VARIABLE ANALYSIS  (Report Section A)")
    print("=" * 60)
    print(f"  Gap lengths : {gap_lengths_hours} hours")
    print(f"  Trials/cell : {n_trials}")
    print(f"  SVD rank    : {rank}")
    print()

    for feat_idx, feat_label in enumerate(FEATURE_LABELS):
        target_col = feat_idx          # station 0 occupies columns 0–3
        print(f"  Feature [{feat_idx}]: {feat_label.replace(chr(10), ' ')}")

        for gap_h in gap_lengths_hours:
            if gap_h >= X_std_clean.shape[0] - 10:
                results[gap_h][feat_label] = {"mean": float("nan"), "std": float("nan")}
                continue

            trial_rmses = []
            for trial in range(n_trials):
                X_masked, mask = inject_burst_missing(
                    X_std_clean, [target_col], hours=gap_h, seed=trial
                )
                X_imp = svt_impute(X_masked, rank=rank)

                row_mask = mask[:, target_col]
                if not row_mask.any():
                    continue
                t = X_std_clean[row_mask, target_col]
                p = X_imp[row_mask, target_col]
                trial_rmses.append(float(np.sqrt(np.nanmean((t - p) ** 2))))

            mean_r = float(np.mean(trial_rmses)) if trial_rmses else float("nan")
            std_r  = float(np.std(trial_rmses))  if trial_rmses else float("nan")
            results[gap_h][feat_label] = {"mean": mean_r, "std": std_r}
            print(f"    gap={gap_h:4d}h  RMSE = {mean_r:.4f} ± {std_r:.4f}")

        print()

    return results


def plot_variable_analysis(va_results: dict) -> None:
    """Two-panel figure: grouped bar chart + RMSE-vs-gap line plot per sensor type."""
    gap_lengths = sorted(va_results.keys())
    feat_labels = FEATURE_LABELS
    colours = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

    means = np.array([[va_results[g][fl]["mean"] for g in gap_lengths] for fl in feat_labels])
    stds  = np.array([[va_results[g][fl]["std"]  for g in gap_lengths] for fl in feat_labels])

    n_feats = len(feat_labels)
    n_gaps  = len(gap_lengths)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left panel: grouped bar chart
    ax = axes[0]
    bar_width = 0.18
    x = np.arange(n_gaps)
    for i, (fl, col) in enumerate(zip(feat_labels, colours)):
        offset = (i - (n_feats - 1) / 2) * bar_width
        short_label = fl.split("\n")[0]
        ax.bar(x + offset, means[i], bar_width,
               yerr=stds[i], capsize=3,
               label=short_label, color=col, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{g}h" for g in gap_lengths])
    ax.set_xlabel("Gap Duration")
    ax.set_ylabel("RMSE  (standardised scale)")
    ax.set_title("Imputation RMSE by Sensor Type\n(grouped by gap length)")
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(axis="y", alpha=0.35)

    # Right panel: line plot
    ax2 = axes[1]
    markers = ["o", "s", "^", "D"]
    for i, (fl, col, mk) in enumerate(zip(feat_labels, colours, markers)):
        short_label = fl.split("\n")[0]
        ax2.plot(gap_lengths, means[i], marker=mk, color=col, lw=2, label=short_label)
        ax2.fill_between(gap_lengths,
                         means[i] - stds[i],
                         means[i] + stds[i],
                         alpha=0.12, color=col)
    ax2.set_xlabel("Gap Duration (hours)")
    ax2.set_ylabel("RMSE  (standardised scale)")
    ax2.set_title("RMSE Degradation per Sensor Type\nas Gap Length Increases")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.35)

    plt.tight_layout()
    plt.savefig("variable_analysis.png", dpi=120)
    plt.show()
    print("  Saved: variable_analysis.png")

    # Print ranking table
    print("\n  Sensor ranking by RMSE (lower = easier to reconstruct):")
    print(f"  {'Sensor':<25}  {'3h':>8}  {'24h':>8}  {'72h':>8}")
    print("  " + "-" * 55)
    for i, fl in enumerate(feat_labels):
        short = fl.split("\n")[0]
        vals  = "  ".join(f"{means[i, j]:>8.4f}" for j in range(n_gaps))
        print(f"  {short:<25}  {vals}")

---
## Rank Selection Analysis (Report Section C)

Sweeps over candidate ranks $r = 1, 2, \ldots, 12$ and records imputation RMSE for a fixed 24-hour burst gap. The resulting RMSE-vs-rank curve reveals how many principal components are needed to capture the dominant agricultural dynamics.

In [ ]:
def rank_selection_analysis(X_std_clean: np.ndarray,
                            rank_range: range,
                            hours_burst: int = 24) -> dict[int, float]:
    """
    Sweep over candidate ranks and record imputation RMSE for a fixed
    24-hour burst gap in station 0. Returns a {rank: rmse} mapping.
    """
    n_feat = len(FEATURES)
    station0_cols = list(range(n_feat))
    X_masked, mask = inject_burst_missing(X_std_clean, station0_cols, hours=hours_burst)

    rmse_vs_rank = {}
    print("=" * 60)
    print("RANK SELECTION ANALYSIS")
    print("=" * 60)
    for r in rank_range:
        X_imp = svt_impute(X_masked, rank=r)
        true_vals = X_std_clean[mask]
        imp_vals  = X_imp[mask]
        r_val = float(np.sqrt(np.nanmean((true_vals - imp_vals) ** 2)))
        rmse_vs_rank[r] = r_val
        print(f"  rank={r:3d}  RMSE={r_val:.4f}")
    return rmse_vs_rank


def plot_rank_vs_rmse(rmse_vs_rank: dict[int, float]) -> None:
    ranks = list(rmse_vs_rank.keys())
    rmses = [rmse_vs_rank[r] for r in ranks]
    plt.figure(figsize=(7, 4))
    plt.plot(ranks, rmses, "o-", color="steelblue", linewidth=2)
    plt.xlabel("Rank  r")
    plt.ylabel("RMSE  (standardised)")
    plt.title("RMSE vs. Rank (24-hour burst gap, Station 1)")
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig("rank_vs_rmse.png", dpi=120)
    plt.show()
    print("  Saved: rank_vs_rmse.png")

---
## Temporal Window Stress Test (Report Section B)

Gap lengths from 3h to 120h are tested with 5 random start positions each. The heuristic breakdown point is defined as the first gap duration at which mean RMSE exceeds twice the 3-hour baseline.

In [ ]:
def temporal_window_stress_test(X_std_clean: np.ndarray,
                                rank: int,
                                gap_lengths_hours: list[int] | None = None,
                                n_trials: int = 5) -> dict[int, dict]:
    """
    Sweep over a range of contiguous gap lengths and record how imputation
    quality degrades as the gap grows.

    Returns
    -------
    results : {gap_hours: {"mean_rmse", "std_rmse", "temp_mean", "wind_mean"}}
    """
    if gap_lengths_hours is None:
        gap_lengths_hours = [3, 6, 12, 24, 36, 48, 72, 96, 120]

    n_feat = len(FEATURES)
    station0_cols = list(range(n_feat))
    temp_col, wind_col = 0, 3

    results = {}

    print("=" * 60)
    print("TEMPORAL WINDOW STRESS TEST  (Report Section B)")
    print("=" * 60)
    print(f"  Gap lengths tested (hours): {gap_lengths_hours}")
    print(f"  Trials per gap length     : {n_trials}")
    print(f"  SVD rank                  : {rank}")
    print()

    for gap_h in gap_lengths_hours:
        if gap_h >= X_std_clean.shape[0] - 10:
            print(f"  gap={gap_h:4d}h  skipped (not enough rows)")
            continue

        trial_rmses, trial_temp, trial_wind = [], [], []

        for trial in range(n_trials):
            X_masked, mask = inject_burst_missing(
                X_std_clean, station0_cols, hours=gap_h, seed=trial
            )
            X_imp = svt_impute(X_masked, rank=rank)

            true_vals = X_std_clean[mask]
            imp_vals  = X_imp[mask]
            trial_rmses.append(float(np.sqrt(np.nanmean((true_vals - imp_vals) ** 2))))

            row_mask_temp = mask[:, temp_col]
            if row_mask_temp.any():
                t = X_std_clean[row_mask_temp, temp_col]
                p = X_imp[row_mask_temp, temp_col]
                trial_temp.append(float(np.sqrt(np.nanmean((t - p) ** 2))))

            row_mask_wind = mask[:, wind_col]
            if row_mask_wind.any():
                t = X_std_clean[row_mask_wind, wind_col]
                p = X_imp[row_mask_wind, wind_col]
                trial_wind.append(float(np.sqrt(np.nanmean((t - p) ** 2))))

        mean_r = float(np.mean(trial_rmses))
        std_r  = float(np.std(trial_rmses))
        mean_t = float(np.mean(trial_temp)) if trial_temp else float("nan")
        mean_w = float(np.mean(trial_wind)) if trial_wind else float("nan")

        results[gap_h] = {
            "mean_rmse": mean_r, "std_rmse": std_r,
            "temp_mean": mean_t, "wind_mean": mean_w,
        }

        print(f"  gap={gap_h:4d}h  |  overall RMSE = {mean_r:.4f} ± {std_r:.4f}"
              f"  |  temp = {mean_t:.4f}  |  wind = {mean_w:.4f}")

    # Heuristic breakdown point
    baseline = results.get(gap_lengths_hours[0], {}).get("mean_rmse", None)
    if baseline is not None:
        threshold = 2.0 * baseline
        breakdown = next((g for g, v in results.items() if v["mean_rmse"] >= threshold), None)
        if breakdown:
            print(f"\n  >> Estimated breakdown point: ≈{breakdown}h "
                  f"(RMSE doubles relative to {gap_lengths_hours[0]}-hour baseline)")
        else:
            print(f"\n  >> RMSE never doubled within the tested range; "
                  f"correlation holds up to {max(results.keys())}h.")

    return results


def plot_temporal_window_stress(tw_results: dict[int, dict]) -> None:
    """Plot overall RMSE (mean ± std) and per-feature RMSE vs gap duration."""
    gaps      = sorted(tw_results.keys())
    means_arr = np.array([tw_results[g]["mean_rmse"] for g in gaps])
    stds_arr  = np.array([tw_results[g]["std_rmse"]  for g in gaps])
    temps     = [tw_results[g]["temp_mean"] for g in gaps]
    winds     = [tw_results[g]["wind_mean"] for g in gaps]
    gaps_arr  = np.array(gaps)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: overall RMSE with ±std band
    ax = axes[0]
    ax.plot(gaps_arr, means_arr, "o-", color="steelblue", lw=2, label="Mean RMSE")
    ax.fill_between(gaps_arr, means_arr - stds_arr, means_arr + stds_arr,
                    alpha=0.25, color="steelblue", label="±1 std")
    ax.axhline(2 * means_arr[0], color="red", ls="--", lw=1.2,
               label=f"Breakdown threshold (2x {gaps[0]}h)")
    ax.set_xlabel("Gap Duration (hours)")
    ax.set_ylabel("RMSE  (standardised scale)")
    ax.set_title("Overall RMSE vs. Gap Length\n(Station 1, mean of trials)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.35)

    # Right: temperature vs wind speed
    ax2 = axes[1]
    ax2.plot(gaps_arr, temps, "s-", color="tomato",     lw=2, label="Temperature")
    ax2.plot(gaps_arr, winds, "^-", color="darkorange", lw=2, label="Wind Speed")
    ax2.set_xlabel("Gap Duration (hours)")
    ax2.set_ylabel("Feature RMSE  (standardised scale)")
    ax2.set_title("Temperature vs. Wind Speed RMSE\nvs. Gap Length")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.35)

    plt.tight_layout()
    plt.savefig("temporal_window_stress.png", dpi=120)
    plt.show()
    print("  Saved: temporal_window_stress.png")

---
## Plotting Utility – Imputation Preview

In [ ]:
def plot_imputation_sample(X_std_clean: np.ndarray,
                           X_std_imputed: np.ndarray,
                           mask: np.ndarray,
                           col: int = 0,
                           col_label: str = "sicaklik_2m_mean (s1)") -> None:
    """Plot observed, ground-truth, and imputed values for one column."""
    N = X_std_clean.shape[0]
    t = np.arange(N)
    obs   = X_std_clean[:, col].copy(); obs[mask[:, col]] = np.nan
    truth = X_std_clean[:, col]
    imp   = X_std_imputed[:, col]

    plt.figure(figsize=(12, 3))
    plt.plot(t, truth, "k-",  alpha=0.3, label="Ground truth")
    plt.plot(t, obs,   "b.",  ms=2,      label="Observed")
    gap_rows = np.where(mask[:, col])[0]
    if len(gap_rows):
        plt.plot(gap_rows, imp[gap_rows], "r-", lw=2, label="Imputed (SVT)")
    plt.legend(fontsize=8)
    plt.title(f"Imputation preview – {col_label}")
    plt.xlabel("Time step (hours)")
    plt.ylabel("Standardised value")
    plt.tight_layout()
    plt.savefig("imputation_preview.png", dpi=120)
    plt.show()
    print("  Saved: imputation_preview.png")

---
## Main Pipeline – End-to-End Execution

This cell runs the full pipeline in order:

1. Load & align station data (Task 0)
2. Standardise (Task 1)
3. Full SVT imputation of the original dataset (Task 3)
4. Orthogonality verification (Task 2)
5. Stress tests — Scenarios A / B / C (Task 4)
6. Variable analysis (Report §A)
7. Temporal window stress test (Report §B)
8. Rank selection sweep (Report §C)
9. Imputation preview plot
10. Summary printout

### **To use with the Blind Test Set**, change only the file paths in `station_paths` below.

In [ ]:
# ── 0. File paths – change these for the Blind Test ──────────────────────────
station_paths = [
    "station1.csv",
    "station2.csv",
    "station3.csv",
]

# ── Task 0 + 1: Load, align, standardise ─────────────────────────────────────
print("=" * 60)
print("TASK 0 + 1 – Loading, aligning, standardising data")
print("=" * 60)

merged = load_and_align(station_paths)
n_stations = len(station_paths)
print(f"  Merged matrix shape after alignment: {merged.shape}")
print(f"  Columns: {merged.columns.tolist()}")

X_raw = merged.values.astype(float)          # N × 12, NaNs possible
X_std, col_mean, col_std = standardise(X_raw)
print(f"  NaN count in standardised matrix: {np.isnan(X_std).sum()}")

# Build 'clean' subset: rows where station 0 is fully observed
n_feat = len(FEATURES)
s0_cols = list(range(n_feat))
fully_observed_mask = ~np.isnan(X_std[:, s0_cols]).any(axis=1)
X_std_clean_subset  = X_std[fully_observed_mask]
print(f"  Fully-observed rows (station 0): {fully_observed_mask.sum()} / {X_std.shape[0]}")

In [ ]:
# ── Task 3: SVT imputation of the original dataset ────────────────────────────
print("=" * 60)
print("TASK 3 – SVT imputation of the original dataset")
print("=" * 60)

DEFAULT_RANK = 4     # sensible default; tuned via rank analysis below
X_std_imputed = svt_impute(X_std, rank=DEFAULT_RANK)
X_imputed_orig_scale = unstandardise(X_std_imputed, col_mean, col_std)
print(f"  Imputed matrix shape: {X_imputed_orig_scale.shape}")
print(f"  Any remaining NaN: {np.isnan(X_imputed_orig_scale).any()}")

In [ ]:
# ── Task 2: SVD orthogonality verification ────────────────────────────────────
print("=" * 60)
print("TASK 2 – SVD orthogonality verification")
print("=" * 60)

# Run on a 200-row slice to keep runtime manageable
sample = X_std_imputed[:200]
U_s, sigma_s, Vt_s = my_svd(sample)
verify_orthogonality(U_s, Vt_s)
print(f"  Top-5 singular values: {sigma_s[:5]}")

In [ ]:
# ── Task 4: Stress tests ──────────────────────────────────────────────────────
stress_results = run_stress_tests(
    X_std_clean_subset,
    col_mean[:n_feat],
    col_std[:n_feat],
    n_stations=n_stations,
    rank=DEFAULT_RANK,
)

In [ ]:
# ── Report Section A: Variable analysis ──────────────────────────────────────
va_results = variable_analysis(
    X_std_clean_subset,
    rank=DEFAULT_RANK,
    gap_lengths_hours=[3, 24, 72],
    n_trials=5,
)
plot_variable_analysis(va_results)

In [ ]:
# ── Report Section B: Temporal window stress test ─────────────────────────────
tw_results = temporal_window_stress_test(
    X_std_clean_subset,
    rank=DEFAULT_RANK,
    gap_lengths_hours=[3, 6, 12, 24, 36, 48, 72, 96, 120],
    n_trials=5,
)
plot_temporal_window_stress(tw_results)

In [ ]:
# ── Report Section C: Rank selection sweep ────────────────────────────────────
max_possible_rank = min(X_std_clean_subset.shape) - 1
rank_range = range(1, min(max_possible_rank + 1, 13))
rmse_vs_rank = rank_selection_analysis(X_std_clean_subset, rank_range)
plot_rank_vs_rmse(rmse_vs_rank)

In [ ]:
# ── Imputation preview plot ───────────────────────────────────────────────────
X_masked_preview, mask_preview = inject_burst_missing(
    X_std_clean_subset, s0_cols, hours=24, seed=1
)
X_imp_preview = svt_impute(X_masked_preview, rank=DEFAULT_RANK)
plot_imputation_sample(
    X_std_clean_subset, X_imp_preview, mask_preview,
    col=0, col_label="sicaklik_2m_mean (Station 1)"
)

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("SUMMARY")
print("=" * 60)

best_rank = min(rmse_vs_rank, key=rmse_vs_rank.get)
print(f"  Best rank (24-h burst test): r={best_rank}  "
      f"RMSE={rmse_vs_rank[best_rank]:.4f}")

for scenario, vals in stress_results.items():
    print(f"  {scenario:22s}  overall={vals['overall']:.4f}  "
          f"temp={vals['temperature_rmse']:.4f}  "
          f"wind={vals['wind_rmse']:.4f}")

print("\n  Temporal Window Stress Test results:")
baseline_rmse = tw_results[min(tw_results)]["mean_rmse"]
for gap_h, vals in sorted(tw_results.items()):
    ratio = vals["mean_rmse"] / baseline_rmse if baseline_rmse > 0 else float("nan")
    print(f"    gap={gap_h:4d}h  RMSE={vals['mean_rmse']:.4f}  "
          f"(×{ratio:.2f} vs baseline)  "
          f"temp={vals['temp_mean']:.4f}  wind={vals['wind_mean']:.4f}")

print("\nDone. Figures saved to working directory.")